   
# Genie Feedback Extraction

This notebook **incrementally** extracts user feedback (POSITIVE / NEGATIVE / no rating) from the Genie space using the Databricks **Genie Conversation REST API**.

**How it works:**
1. Reads the **Lookback Days** widget to determine the time window (default: 7 days)
2. Lists conversations and filters to only those created within the lookback window
3. Fetches messages within each filtered conversation, skipping any outside the window
4. Extracts `feedback.rating`, user question, Genie response, and generated SQL
5. **MERGE**s results into the target Delta table — inserts new rows, updates changed ratings (idempotent)

In [0]:
import requests
import json
import time
from datetime import datetime, timedelta
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration — change the Genie Space ID widget at the top to target
# a different Genie space
# ---------------------------------------------------------------------------
SPACE_ID = dbutils.widgets.get("space_id")

# Incremental window — only fetch conversations/messages from the last N days
LOOKBACK_DAYS = int(dbutils.widgets.get("lookback_days"))
cutoff_dt = datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)
CUTOFF_EPOCH_MS = int(cutoff_dt.timestamp() * 1000)
print(f"Lookback: {LOOKBACK_DAYS} day(s) — fetching data since {cutoff_dt.strftime('%Y-%m-%d %H:%M:%S')} UTC")

# Auth (uses the current notebook session token)
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = (
    dbutils.notebook.entry_point
    .getDbutils().notebook().getContext().apiToken().get()
)

BASE_URL = f"https://{workspace_url}/api/2.0/genie/spaces/{SPACE_ID}"
HEADERS = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

In [0]:
resp = requests.get(f"{BASE_URL}/conversations", headers=HEADERS)
resp.raise_for_status()

all_conversations = resp.json().get("conversations", [])

# Filter to conversations within the lookback window
conversations = [
    c for c in all_conversations
    if c["created_timestamp"] >= CUTOFF_EPOCH_MS
]

print(f"Found {len(all_conversations)} total conversation(s) in space {SPACE_ID}")
print(f"Filtered to {len(conversations)} conversation(s) within the last {LOOKBACK_DAYS} day(s)\n")
for c in conversations:
    ts = datetime.fromtimestamp(c['created_timestamp'] / 1000).strftime('%Y-%m-%d %H:%M:%S')
    print(f"  [{ts}] {c['conversation_id']}  —  {c['title'][:90]}")

In [0]:
# Fetch the Genie space name
space_resp = requests.get(BASE_URL, headers=HEADERS)
space_resp.raise_for_status()
space_name = space_resp.json().get("title", "")
print(f"Space: {space_name} ({SPACE_ID})")
print(f"Incremental window: last {LOOKBACK_DAYS} day(s)\n")

rows = []
skipped_messages = 0

for conv in conversations:
    conv_id = conv["conversation_id"]
    conv_title = conv["title"]

    resp = requests.get(
        f"{BASE_URL}/conversations/{conv_id}/messages", headers=HEADERS
    )
    resp.raise_for_status()
    messages = resp.json().get("messages", [])

    for msg in messages:
        # Skip messages outside the lookback window
        if msg["created_timestamp"] < CUTOFF_EPOCH_MS:
            skipped_messages += 1
            continue

        # Extract Genie's text response and SQL from attachments
        genie_text = None
        genie_sql = None
        for att in msg.get("attachments") or []:
            if "text" in att:
                genie_text = att["text"].get("content")
            if "query" in att:
                genie_sql = att["query"].get("query")

        feedback = msg.get("feedback")
        rating = feedback.get("rating") if feedback else None

        rows.append({
            "genie_space_id": SPACE_ID,
            "genie_space_name": space_name,
            "conversation_id": conv_id,
            "conversation_title": conv_title,
            "message_id": msg.get("message_id"),
            "user_id": msg.get("user_id"),
            "timestamp": datetime.fromtimestamp(
                msg["created_timestamp"] / 1000
            ).strftime("%Y-%m-%d %H:%M:%S"),
            "user_question": msg.get("content", ""),
            "genie_response": genie_text,
            "genie_sql": genie_sql,
            "feedback_rating": rating,
            "status": msg.get("status"),
        })

    time.sleep(0.2)  # Throttle to avoid rate-limiting

# Build DataFrame
df_feedback = pd.DataFrame(rows)

# Summary
total = len(df_feedback)
positive = (df_feedback["feedback_rating"] == "POSITIVE").sum() if total > 0 else 0
negative = (df_feedback["feedback_rating"] == "NEGATIVE").sum() if total > 0 else 0
no_fb = df_feedback["feedback_rating"].isna().sum() if total > 0 else 0

print(f"Messages extracted:  {total}")
print(f"Messages skipped (outside window): {skipped_messages}")
print(f"  POSITIVE:    {positive}")
print(f"  NEGATIVE:    {negative}")
print(f"  No feedback: {no_fb}")
print()

if total > 0:
    display(df_feedback)
else:
    print("No new messages found in the lookback window.")

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
TABLE   = dbutils.widgets.get("table")
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

# Ensure catalog and schema exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Create the target table if it doesn't exist
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        genie_space_id STRING,
        genie_space_name STRING,
        conversation_id STRING,
        conversation_title STRING,
        message_id STRING,
        user_id LONG,
        timestamp STRING,
        user_question STRING,
        genie_response STRING,
        genie_sql STRING,
        feedback_rating STRING,
        status STRING
    )
    USING DELTA
""")

if df_feedback.empty:
    print(f"No new messages in the last {LOOKBACK_DAYS} day(s) — nothing to upsert.")
else:
    # Convert pandas DataFrame to Spark
    df_spark = spark.createDataFrame(df_feedback)

    # Create a temp view for MERGE
    df_spark.createOrReplaceTempView("feedback_staging")

    # MERGE — idempotent: inserts new rows, updates feedback_rating if it changed
    # Uses IS DISTINCT FROM to correctly handle NULL <-> non-NULL transitions
    result = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS target
        USING feedback_staging AS source
        ON target.message_id = source.message_id
        WHEN MATCHED AND target.feedback_rating IS DISTINCT FROM source.feedback_rating THEN UPDATE SET
            target.feedback_rating = source.feedback_rating
        WHEN NOT MATCHED THEN INSERT *
    """)

    display(result)
    print(f"\nTarget table: {TARGET_TABLE}")

   
## Notes

| Item | Detail |
|---|---|
| **Permission required** | `CAN MANAGE` on the Genie space (needed to list conversations across all users) |
| **Incremental window** | Controlled by the **Lookback Days** dropdown (1, 3, 7, 14, 30, 90, or 365 days). Only conversations and messages within this window are fetched. |
| **Idempotent writes** | The `MERGE` on `message_id` inserts new rows and updates only when `feedback_rating` changes (null-safe via `IS DISTINCT FROM`). Re-runs are safe. |
| **Feedback values** | `POSITIVE`, `NEGATIVE`, or `null` (no feedback submitted) |
| **Rate-limiting** | A `time.sleep(0.2)` pause is added between API calls; increase if you hit throttling on large spaces |
| **Different space** | Change the **space_id** widget at the top of the notebook |
| **API endpoints used** | `GET /api/2.0/genie/spaces/{space_id}/conversations` and `GET .../conversations/{conversation_id}/messages` |